In [1]:
import sys
from google.colab import drive

# 1. Mount the drive
drive.mount('/content/drive')

# 2. Add your repository folder to the Python path
# Replace 'YourRepoName' with the actual folder name in your Drive
repo_path = '/content/drive/MyDrive/Snake'
if repo_path not in sys.path:
    sys.path.append(repo_path)

import os

# Use the same repo_path we defined earlier
os.chdir(repo_path)

# Verify we are in the right place
print("Current Directory:", os.getcwd()) 


Mounted at /content/drive
Current Directory: /content/drive/Othercomputers/My laptop/Snake


In [7]:
import os
import sys
import subprocess
import time
from train import train

def run_train_in_subprocess(use_gpu):
    env = os.environ.copy()
    if use_gpu:
        env.pop("CUDA_VISIBLE_DEVICES", None)
    else:
        env["CUDA_VISIBLE_DEVICES"] = "-1"

    code = """
import time
from train import train
start = time.time()
train()
print("TRAIN_RUNTIME", time.time() - start)
"""
    proc = subprocess.run(
        [sys.executable, "-c", code],
        env=env,
        capture_output=True,
        text=True,
    )
    if proc.returncode != 0:
        raise RuntimeError(
            f"train() failed on {'GPU' if use_gpu else 'CPU'}:\\n{proc.stderr}"
        )

    for line in proc.stdout.splitlines()[::-1]:
        if line.startswith("TRAIN_RUNTIME"):
            return float(line.split()[1])

    raise RuntimeError("Could not parse runtime from subprocess output")

cpu_time = run_train_in_subprocess(use_gpu=False)
gpu_time = run_train_in_subprocess(use_gpu=True)

print(f"CPU runtime: {cpu_time:.2f} s")
print(f"GPU runtime: {gpu_time:.2f} s")

RuntimeError: train() failed on CPU:\nTraceback (most recent call last):
  File "<string>", line 5, in <module>
  File "/content/drive/Othercomputers/My laptop/Snake/train.py", line 168, in train
    agent = brain.actor_cnn(input_size=input_size, output_size=3, hidden_size=config.network_parameters.hidden_size,
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: actor_cnn.__init__() missing 1 required positional argument: 'state_shape'
